# Module 12: Probabilistic Data Structures — Interactive Laboratory

Every cell below runs the module's **real** implementation from
`project_solution/probabilistic_structures.py`. Nothing here prints a claim it has not verified.

What you will do:

1. Load the engine and inspect what it actually exports.
2. Run its primary workflow and check the assertions that define correctness.
3. **Commit to a prediction**, then run the cell that tests it.
4. Measure a property rather than asserting one.
5. Fix a deliberately broken cell in place.

> The code in cells 4, 6 and 8 is lifted from this module's own test suite, so it
> cannot drift from the implementation. If the API changes, those tests fail
> first and this notebook is regenerated from them.


## 1. Load the engine and introspect it

Rather than trusting a hardcoded list of class names, ask the module what it
actually contains.


In [ ]:
import inspect
import sys
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve() / 'project_solution'))
import probabilistic_structures

classes = [n for n, o in inspect.getmembers(probabilistic_structures, inspect.isclass)
           if o.__module__ == 'probabilistic_structures']
functions = [n for n, o in inspect.getmembers(probabilistic_structures, inspect.isfunction)
             if o.__module__ == 'probabilistic_structures']

print('module   : probabilistic_structures')
print(f'classes  : {classes}')
print(f'functions: {functions}')
print()
for name in classes:
    obj = getattr(probabilistic_structures, name)
    try:
        sig = inspect.signature(obj.__init__)
        params = [p for p in sig.parameters if p != 'self']
    except (TypeError, ValueError):
        params = ['<builtin>']
    print(f'  {name}({", ".join(params)})')

## 2. Baseline: Bloom filter zero false negatives

This is the module's own `test_bloom_filter_zero_false_negatives` — real instantiation, real calls, real
assertions. If it runs clean, the property it encodes holds.


In [ ]:
from probabilistic_structures import (
    BloomFilter,
    CountMinSketch,
)

bf = BloomFilter(expected_items=1000, fp_rate=0.01)
added_items = [f"user_{i}" for i in range(1000)]

for item in added_items:
    bf.add(item)

# Invariant: Every added item MUST be present (Zero False Negatives)
for item in added_items:
    assert item in bf

print('PASSED: test_bloom_filter_zero_false_negatives')

## 3. 🔮 Prediction — commit before you run

A Bloom filter reports an item is present. Predict whether it might be absent, and whether the reverse (reports absent but is present) is also possible.

Write your answer down. An uncommitted guess teaches nothing, because you will
retro-fit it to whatever the next cell prints.

The next cell runs `test_bloom_filter_false_positive_rate_within_bounds`, which tests exactly this property.


In [ ]:
bf = BloomFilter(expected_items=2000, fp_rate=0.05)
for i in range(2000):
    bf.add(f"member_{i}")

# Test 2000 non-members
false_positives = 0
test_non_members = 2000
for i in range(test_non_members):
    if f"non_member_{i}" in bf:
        false_positives += 1

measured_fp_rate = false_positives / test_non_members
# Measured FP rate should be close to 0.05 (allowing tolerance up to 0.08)
assert measured_fp_rate < 0.08

print('PASSED: test_bloom_filter_false_positive_rate_within_bounds')

## 4. Measure it: Count min sketch never underestimates

An assertion tells you a property holds. A measurement tells you *how much*.
This cell runs `test_count_min_sketch_never_underestimates` and times it.


In [ ]:
import time

_t0 = time.perf_counter()

cms = CountMinSketch(width=500, depth=5)

# Feed item counts
for _ in range(42):
    cms.increment("apple")
for _ in range(15):
    cms.increment("banana")

# Invariant: Estimate must be >= actual count (never underestimates)
assert cms.estimate("apple") >= 42
assert cms.estimate("banana") >= 15
assert cms.estimate("cherry") == 0

_elapsed = (time.perf_counter() - _t0) * 1000
print('PASSED: test_count_min_sketch_never_underestimates')
print(f'wall clock: {_elapsed:.2f} ms')

## 5. 🛠️ Fix this cell — it is deliberately broken

The cell below asserts something **false** about the real object. Read the
failure, work out the true value from the module's actual behaviour, and correct
the expected number.

Do not delete the assertion. The point is to make it pass by knowing the answer.


In [ ]:
# DELIBERATELY BROKEN - fix the expected value below.
# Hint: print the real value first, then decide what the assertion should say.

exports = [n for n in dir(probabilistic_structures) if not n.startswith('_')]
print(f'actual export count: {len(exports)}')
print(f'actual exports     : {exports}')

EXPECTED_EXPORT_COUNT = 999      # <-- wrong on purpose. Replace it.

assert len(exports) == EXPECTED_EXPORT_COUNT, (
    f'expected {EXPECTED_EXPORT_COUNT} exports, found {len(exports)}. '
    'Read the printed value above and correct the constant.'
)
print('Fixed - assertion now reflects reality.')

### 🎓 Key takeaways

1. Bloom filters trade a bounded false-positive rate for enormous space savings.
2. False negatives are impossible - that asymmetry is what makes them useful.
3. Sizing is a formula, not a guess: pick m and k from n and your target error rate.

---

**Continue with this module:**

- [README.md](README.md) — the mental model and failure modes
- [PROJECT_GUIDE.md](PROJECT_GUIDE.md) — build it yourself, in 3 tiers
- [starter/](starter/) — your stubs; run the tests from there to grade yourself
- [debug_lab/SYMPTOMS.md](debug_lab/SYMPTOMS.md) — diagnose planted bugs from the symptom
- [TROUBLESHOOTING_AND_EDGE_CASES.md](TROUBLESHOOTING_AND_EDGE_CASES.md) — real errors, real causes
- [SELF_ASSESSMENT_AND_CHALLENGES.md](SELF_ASSESSMENT_AND_CHALLENGES.md) — quiz and diagnostics
